In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import json

### Significance Test 1: Selectiveness

**"For each censoring method (feature noise, label noise, omission), does sensitive performance degrade significantly more than non-sensitive performance at each noise level?"**

- Null hypothesis (H₀): Sensitive and non-sensitive regions degrade equally (no selective effect)
- Alternative hypothesis (H₁): Sensitive region degrades more than non-sensitive region (selective censoring works)

For each method, at each noise level, where the baseline is unmodified, raw data:
- sensitive_degradation = (baseline_sensitive - noisy_sensitive) / baseline_sensitive * 100
- non_sensitive_degradation = (baseline_non_sensitive - noisy_non_sensitive) / baseline_non_sensitive * 100

Test: sensitive_degradation > non_sensitive_degradation

In [ ]:
def test_selective_censoring(history_file, censor_type, censor_intervals, 
                           censor_region, metric='corr'):
    """
    Test selective censoring for one method with its specific noise intervals
    """
    with open(history_file, 'r') as f:
        history_data = json.load(f)
    
    if metric == 'corr':
        all_lower = np.array(history_data[5])  # lower region
        all_upper = np.array(history_data[6])  # upper region
    else:
        all_lower = np.array(history_data[2])
        all_upper = np.array(history_data[3])
    
    # assign sensitive vs non-sensitive based on censor_region
    if censor_region == 'above':
        sensitive_data = all_upper
        non_sensitive_data = all_lower
    elif censor_region == 'below':
        sensitive_data = all_lower
        non_sensitive_data = all_upper
    else:
        raise ValueError("censor_region must be 'above' or 'below'")
    
    # baseline (0% noise)
    baseline_sensitive = sensitive_data[:, 0]
    baseline_non_sensitive = non_sensitive_data[:, 0]
    
    results = []
       
    for censor_idx in range(1, len(censor_intervals)):
        # calculate percentage degradation
        sens_deg = (baseline_sensitive - sensitive_data[:, censor_idx]) / np.abs(baseline_sensitive) * 100
        non_sens_deg = (baseline_non_sensitive - non_sensitive_data[:, censor_idx]) / np.abs(baseline_non_sensitive) * 100
        
        # get actual correlation values for verification
        sens_corr_avg = np.mean(sensitive_data[:, censor_idx])
        non_sens_corr_avg = np.mean(non_sensitive_data[:, censor_idx])
        baseline_sens_avg = np.mean(baseline_sensitive)
        baseline_non_sens_avg = np.mean(baseline_non_sensitive)
        
        # wilcoxon test
        try:
            statistic, p_value = wilcoxon(sens_deg, non_sens_deg, alternative='greater')
        except ValueError:
            p_value = 1.0
        
        sig_marker = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
        
        results.append({
            'Method': censor_type,
            'Noise_Level': censor_intervals[censor_idx],
            'Baseline_Sens': f"{baseline_sens_avg:.3f}",
            'Current_Sens': f"{sens_corr_avg:.3f}", 
            'Baseline_NonSens': f"{baseline_non_sens_avg:.3f}",
            'Current_NonSens': f"{non_sens_corr_avg:.3f}",
            'p_value': p_value,
            'p_formatted': f"{p_value:.3f}{sig_marker}",
            'significant': p_value < 0.05
        })
    
    return pd.DataFrame(results)

In [ ]:
censor_region = 'above'

In [ ]:
# Load the 10% omission experiment history
history_file = f'../all_results/gcn_omit_results_split0.1_{censor_region}/history.json'
censor_type = 'omit'
omit_fractions = np.linspace(0, 1, int(1/0.1+1))

df = test_selective_censoring(
    history_file, 
    censor_type, 
    omit_fractions,
    censor_region=censor_region
)
df

In [ ]:
# Load the 50% omission experiment history
history_file = f'../all_results/gcn_omit_results_split0.5_{censor_region}/history.json'
censor_type = 'omit'
omit_fractions = np.linspace(0, 1, int(1/0.1+1))

df = test_selective_censoring(
    history_file, 
    censor_type, 
    omit_fractions,
    censor_region=censor_region
)
df

In [ ]:
# Load the 90% omission experiment history
history_file = f'../all_results/gcn_omit_results_split0.9_{censor_region}/history.json'
censor_type = 'omit'
omit_fractions = np.linspace(0, 1, int(1/0.1+1))

df = test_selective_censoring(
    history_file, 
    censor_type, 
    omit_fractions,
    censor_region=censor_region
)
df

In [ ]:
similarity_intervals = [
    [1.0, 1.0], # no-noise control
    [0.8, 1.0],
    [0.7, 0.8],
    [0.6, 0.7],
    [0.5, 0.6],
    [0.4, 0.5],
    [0.35, 0.4],
    [0.3, 0.35],
    [0.25, 0.3],
    [0.2, 0.25],
    [0.15, 0.2],
    [0.1, 0.15],
    [0.05, 0.1],
    [0, 0.05],
]
sim_scores_strings_list = [f'{interval[0]}-{interval[1]}' for interval in similarity_intervals]

In [ ]:
# Load 10% x-noise experiment history
history_file = f'../all_results/gcn_xnoise_results_split0.1_{censor_region}/history.json'
censor_type = 'xnoise'

df = test_selective_censoring(
    history_file, 
    censor_type, 
    sim_scores_strings_list,
    censor_region=censor_region
)
df

In [ ]:
# Load 50% x-noise experiment history
history_file = f'../all_results/gcn_xnoise_results_split0.5_{censor_region}/history.json'
censor_type = 'xnoise'

df = test_selective_censoring(
    history_file, 
    censor_type, 
    sim_scores_strings_list,
    censor_region=censor_region
)
df

In [ ]:
# Load 90% x-noise experiment history
history_file = f'../all_results/gcn_xnoise_results_split0.9_{censor_region}/history.json'
censor_type = 'xnoise'

df = test_selective_censoring(
    history_file, 
    censor_type, 
    sim_scores_strings_list,
    censor_region=censor_region
)
df

In [ ]:
interval_step = 0.2
start_level = 0
end_level = 5
y_noise_levels = np.linspace(start_level, end_level, int((end_level - start_level) / interval_step + 1))

In [ ]:
# Load 10% y-noise experiment history
history_file = f'../all_results/gcn_ynoise_results_split0.1_{censor_region}/history.json'
censor_type = 'ynoise'

df = test_selective_censoring(
    history_file, 
    censor_type, 
    y_noise_levels,
    censor_region=censor_region
)
df

In [ ]:
# Load 50% y-noise experiment history
history_file = f'../all_results/gcn_ynoise_results_split0.5_{censor_region}/history.json'
censor_type = 'ynoise'

df = test_selective_censoring(
    history_file, 
    censor_type, 
    y_noise_levels,
    censor_region=censor_region
)
df

In [ ]:
# Load 90% y-noise experiment history
history_file = f'../all_results/gcn_ynoise_results_split0.9_{censor_region}/history.json'
censor_type = 'ynoise'

df = test_selective_censoring(
    history_file, 
    censor_type, 
    y_noise_levels,
    censor_region=censor_region
)
df

### Significance Test 2: Effectiveness
**"Between censoring methods, which method most effectively degrades sensitive region performance?"**

- Null hypothesis (H₀): All methods degrade sensitive performance equally
- Alternative hypothesis (H₁): One method degrades sensitive performance significantly more than another

For pairwise comparisons between methods at maximum noise level:

- method1_sensitive_degradation = (baseline_sensitive - max_noise_sensitive) / baseline_sensitive * 100
- method2_sensitive_degradation = (baseline_sensitive - max_noise_sensitive) / baseline_sensitive * 100

Test: method1_sensitive_degradation > method2_sensitive_degradation

In [ ]:
def test_method_effectiveness(history_files, method_names, censor_region, 
                                  metric='corr', censor_split='0.5'):
    """
    Test 2: Compare method effectiveness at low/medium/high intensity levels
    
    Comparison points:
    - Low: X-noise 0.1, Omission 0.1, Label 1.0  
    - Medium: X-noise 0.55, Omission 0.5, Label 2.6
    - High: X-noise 0.975, Omission 1.0, Label 5.0
    """
    
    # Define comparison points for each method
    comparison_points = {
        'Feature Noise': {'Low': 1, 'Medium': 5, 'High': 13},      # indices for x-noise levels
        'Label Noise': {'Low': 5, 'Medium': 13, 'High': 25},       # indices for label levels  
        'Omission': {'Low': 1, 'Medium': 5, 'High': 10}            # indices for omission fractions
    }
    
    results = []
    
    for intensity in ['Low', 'Medium', 'High']:
        method_degradations = {}
        
        # Get degradation for each method at this intensity level
        for hist_file, method_name in zip(history_files, method_names):
            with open(hist_file, 'r') as f:
                history_data = json.load(f)
            
            if metric == 'corr':
                all_lower = np.array(history_data[5])
                all_upper = np.array(history_data[6])
            else:
                all_lower = np.array(history_data[2])
                all_upper = np.array(history_data[3])
            
            # Assign sensitive data based on censor_region
            sensitive_data = all_lower if censor_region == 'below' else all_upper
            
            # Get the appropriate index for this method and intensity
            target_idx = comparison_points[method_name][intensity]
            
            # Calculate degradation
            baseline = sensitive_data[:, 0]
            if target_idx < sensitive_data.shape[1]:
                current = sensitive_data[:, target_idx]
                degradation = (baseline - current) / np.abs(baseline) * 100
                method_degradations[method_name] = degradation
        
        # Compare Feature Noise vs other methods
        if 'Feature Noise' in method_degradations:
            for other_method in ['Label Noise', 'Omission']:
                if other_method in method_degradations:
                    
                    deg_feature = method_degradations['Feature Noise']
                    deg_other = method_degradations[other_method]
                    
                    # Test if Feature Noise is more effective
                    try:
                        _, p_value = wilcoxon(deg_feature, deg_other, alternative='greater')
                    except ValueError:
                        p_value = 1.0
                    
                    results.append({
                        'Intensity': intensity,
                        'Comparison': f"Feature vs {other_method}",
                        'Feature_AvgDeg': f"{np.mean(deg_feature):.1f}%",
                        'Other_AvgDeg': f"{np.mean(deg_other):.1f}%",
                        'Feature_Superior': p_value < 0.05,
                        'p_value': f"{p_value:.3f}"
                    })
    
    return pd.DataFrame(results)

In [ ]:
# with 10% sensitive data
censor_split= 0.1

history_files = [
    f'../all_results/gcn_xnoise_results_split{censor_split}_{censor_region}/history.json',
    f'../all_results/gcn_ynoise_results_split{censor_split}_{censor_region}/history.json', 
    f'../all_results/gcn_omit_results_split{censor_split}_{censor_region}/history.json'
]

method_names = ['Feature Noise', 'Label Noise', 'Omission']

df = test_method_effectiveness(
    history_files, 
    method_names, 
    censor_region=censor_region,
    censor_split=censor_split
)

df

In [ ]:
# with 50% sensitive data
censor_split= 0.5

history_files = [
    f'../all_results/gcn_xnoise_results_split{censor_split}_{censor_region}/history.json',
    f'../all_results/gcn_ynoise_results_split{censor_split}_{censor_region}/history.json', 
    f'../all_results/gcn_omit_results_split{censor_split}_{censor_region}/history.json'
]

method_names = ['Feature Noise', 'Label Noise', 'Omission']

df = test_method_effectiveness(
    history_files, 
    method_names, 
    censor_region=censor_region,
    censor_split=censor_split
)

df

In [ ]:
# with 90% sensitive data
censor_split= 0.9

history_files = [
    f'../all_results/gcn_xnoise_results_split{censor_split}_{censor_region}/history.json',
    f'../all_results/gcn_ynoise_results_split{censor_split}_{censor_region}/history.json', 
    f'../all_results/gcn_omit_results_split{censor_split}_{censor_region}/history.json'
]

method_names = ['Feature Noise', 'Label Noise', 'Omission']

df = test_method_effectiveness(
    history_files, 
    method_names, 
    censor_region=censor_region,
    censor_split=censor_split
)

df